In [1]:
# 00_setup.ipynb -- environment check and plotting/convention self-test

# === Shared configuration (self-contained: one code cell per notebook) ===
from __future__ import annotations
import os, sys, json, math, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Resolve the repository root (the folder that holds both 'results' and 'notebooks').
def _find_root(start: Path) -> Path:
    p = start.resolve()
    for cand in [p, *p.parents]:
        if (cand / "results").is_dir() and (cand / "notebooks").is_dir():
            return cand
    return start.resolve().parent if start.resolve().name == "notebooks" else start.resolve()

ROOT = _find_root(Path.cwd())
DATA, RAW, PROC = ROOT / "data", ROOT / "data" / "raw", ROOT / "data" / "processed"
RESULTS, FIGS, TABS = ROOT / "results", ROOT / "results" / "figures", ROOT / "results" / "tables"
for d in (DATA, RAW, PROC, RESULTS, FIGS, TABS):
    d.mkdir(parents=True, exist_ok=True)

# Load environment variables (.env is git-ignored; never commit secrets).
try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
except Exception:
    pass
LLM_MODELS = {
    "weak":   os.getenv("LLM_MODEL_WEAK", ""),
    "mid":    os.getenv("LLM_MODEL_MID", ""),
    "strong": os.getenv("LLM_MODEL_STRONG", ""),
}
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")

# Plotting convention: seaborn, grayscale, no caption, 600 dpi, saved as PNG and PDF.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(context="paper", style="ticks")
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 600, "savefig.bbox": "tight",
    "font.size": 9, "axes.titlesize": 9, "axes.labelsize": 9,
    "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8,
    "axes.grid": True, "grid.alpha": 0.3, "image.cmap": "Greys",
})

def gray_palette(n: int):
    # Evenly spaced grays (mid -> dark) for categorical series.
    return sns.color_palette("Greys", max(n + 2, 3))[2:n + 2]

def save_fig(fig, name: str):
    # Save a figure as PNG and PDF at 600 dpi; no caption is added here.
    for ext in ("png", "pdf"):
        fig.savefig(FIGS / f"{name}.{ext}", dpi=600, bbox_inches="tight")
    plt.close(fig)

def save_table(df: pd.DataFrame, name: str, index: bool = False):
    # Persist a results table as CSV and (best-effort) LaTeX for the manuscript.
    df.to_csv(TABS / f"{name}.csv", index=index)
    try:
        (TABS / f"{name}.tex").write_text(df.to_latex(index=index))
    except Exception:
        pass

# Fine-grained rating scale (low -> high); used to map labels to an ordinal rank.
RATING_SCALE = ["D","C","CC","CCC-","CCC","CCC+","B-","B","B+","BB-","BB","BB+",
                "BBB-","BBB","BBB+","A-","A","A+","AA-","AA","AA+","AAA"]
RANK = {g: i for i, g in enumerate(RATING_SCALE)}
def to_rank(label):
    # Map a rating label to an ordinal rank; returns np.nan if unmatched.
    if not isinstance(label, str):
        return np.nan
    s = label.strip().upper().replace(" ", "")
    return RANK.get(s, np.nan)

# --- 00: verify the environment and exercise the figure/table conventions ---
import sklearn
print("Python :", sys.version.split()[0])
print("libs   : pandas", pd.__version__, "| numpy", np.__version__,
      "| seaborn", sns.__version__, "| sklearn", sklearn.__version__)
print("ROOT   :", ROOT)
print("models :", LLM_MODELS)
print("API key:", f"set (len={len(OPENAI_API_KEY)})" if OPENAI_API_KEY else "NOT set (LLM notebooks will skip)")

# Convention-check figure: grayscale, no caption, 600 dpi, PNG + PDF. Illustrative only.
rng = np.random.default_rng(SEED)
demo = pd.DataFrame({
    "override": np.repeat([0, 1, 2, 3], 60),
    "score": np.concatenate([rng.normal(m, 0.40, 60) for m in (4.5, 3.9, 3.0, 2.1)]),
})
fig, ax = plt.subplots(figsize=(3.4, 2.6))
sns.boxplot(data=demo, x="override", y="score", ax=ax,
            color="0.75", linecolor="0.2", fliersize=2, width=0.6)
ax.set_xlabel(r"Qualitative-override magnitude $|\Delta|$ (notches)")
ax.set_ylabel("Judge score (1-5)")
sns.despine(ax=ax)
save_fig(fig, "_convention_check")
save_table(demo.groupby("override")["score"].mean().round(3).reset_index(), "_convention_check_means")

made = sorted(p.name for p in FIGS.glob("_convention_check.*"))
assert any(x.endswith(".png") for x in made) and any(x.endswith(".pdf") for x in made)
print("wrote figures:", made)
print("setup OK")


Python : 3.11.15
libs   : pandas 3.0.3 | numpy 2.4.6 | seaborn 0.13.2 | sklearn 1.8.0
ROOT   : C:\Users\User\Downloads\학술\71_ragas한계논문\credit-rating-llm-judge\credit-rating-llm-judge
models : {'weak': 'gpt-4o-mini', 'mid': 'gpt-5.4-mini', 'strong': 'gpt-5.4'}
API key: set (len=56)
wrote figures: ['_convention_check.pdf', '_convention_check.png']
setup OK
